In [17]:
#import libraries
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from uuid import uuid4

In [18]:
    #read data
    df = pd.read_csv('dataset\moratuwa_2014-2018.csv')

In [19]:
# Preprocessing
df['ticket_date'] = pd.to_datetime(df['ticket_date'])
df = df.sort_values('ticket_date')

In [20]:
# information about data
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 6691 entries, 0 to 6690
Data columns (total 4 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   area           6691 non-null   object        
 1   ticket_date    6691 non-null   datetime64[ns]
 2   waste_type     6691 non-null   object        
 3   net_weight_kg  6691 non-null   int64         
dtypes: datetime64[ns](1), int64(1), object(2)
memory usage: 261.4+ KB


In [21]:
#display the top 5 rows of data
df.head()

,area,ticket_date,waste_type,net_weight_kg
0,Moratuwa MC,2014-01-12,Sorted Organic Waste,47230
1,Moratuwa MC,2014-02-12,Sorted Organic Waste,54670
2,Moratuwa MC,2014-03-12,Sorted Organic Waste,43670
3,Moratuwa MC,2014-04-12,Sorted Organic Waste,40410
4,Moratuwa MC,2014-05-12,Sorted Organic Waste,34590


In [22]:
#display the bottom 5 rows of data
df.tail()

,area,ticket_date,waste_type,net_weight_kg
6687,Moratuwa MC,2018-12-11,MSW,10370
6689,Moratuwa MC,2018-12-11,Soil With Waste,9290
6685,Moratuwa MC,2018-12-11,Bulky Waste,7190
6686,Moratuwa MC,2018-12-11,Industrial Waste,6210
6690,Moratuwa MC,2018-12-11,Sorted Organic Waste,83910


In [23]:
#value counts on waste_type
df['waste_type'].value_counts()

waste_type
MSW                       1307
Sorted Organic Waste      1208
Slaghter House Waste      1105
Bulky Waste               1022
Industrial Waste           842
Soil With Waste            777
Sanitary Waste             236
C&D Waste                  102
Indutrial Sludge Waste      41
Soil                        16
Saw Dust                    14
Polythyne & Regiform        11
Mesuring                     5
Wood Debris                  3
Special Waste                1
Wood Trank                   1
Name: count, dtype: int64

In [24]:
df['area'].value_counts()

area
Moratuwa MC    6691
Name: count, dtype: int64

In [25]:
# define waste types
waste_types = ['MSW', 'Sorted Organic Waste', 'Slaghter House Waste', 'Bulky Waste', 
               'Industrial Waste', 'Soil With Waste', 'Sanitary Waste', 'C&D Waste', 
               'Indutrial Sludge Waste', 'Soil', 'Saw Dust', 'Polythyne & Regiform', 
               'Mesuring', 'Wood Debris', 'Special Waste', 'Wood Trank']

In [ ]:
# Aggregate by date, keeping waste_type
daily_data = df.groupby(['ticket_date', 'waste_type'])['net_weight_kg'].sum().unstack(fill_value=0).reset_index()
daily_data = daily_data.set_index('ticket_date')

# Filter for 2018 data
daily_data_2018 = daily_data[daily_data.index.year == 2018]

# Add time-based features
daily_data_2018['day_of_week'] = daily_data_2018.index.dayofweek
daily_data_2018['month'] = daily_data_2018.index.month
daily_data_2018['day_of_year'] = daily_data_2018.index.dayofyear

# Total net_weight_kg for univariate forecasting
daily_data_2018['total_weight'] = daily_data_2018[waste_types].sum(axis=1)

# Add Gaussian noise to total_weight
np.random.seed(42)  # For reproducibility
noise = np.random.normal(loc=0, scale=0.1 * daily_data_2018['total_weight'].std(), size=len(daily_data_2018))
daily_data_2018['total_weight'] = daily_data_2018['total_weight'] + noise
daily_data_2018['total_weight'] = daily_data_2018['total_weight'].clip(lower=0)  # Ensure no negative weights

In [ ]:
# Outlier treatment using IQR method
def detect_outliers_iqr(data, column):
    Q1 = data[column].quantile(0.25)
    Q3 = data[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    return lower_bound, upper_bound

# Cap outliers
lower_iqr, upper_iqr = detect_outliers_iqr(daily_data_2018, 'total_weight')
daily_data_2018['total_weight'] = daily_data_2018['total_weight'].clip(lower=lower_iqr, upper=upper_iqr)


<p style="font-family: Calibri, serif; text-align: left;
          font-size: 28px; letter-spacing: .85px; color: #ffffff;">Linear Regression</p>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Read data
df = pd.read_csv('dataset\moratuwa_2014-2018.csv')

# Preprocessing
df['ticket_date'] = pd.to_datetime(df['ticket_date'])
df = df.sort_values('ticket_date')

# Define waste types
waste_types = ['MSW', 'Sorted Organic Waste', 'Slaghter House Waste', 'Bulky Waste', 
               'Industrial Waste', 'Soil With Waste', 'Sanitary Waste', 'C&D Waste', 
               'Indutrial Sludge Waste', 'Soil', 'Saw Dust', 'Polythyne & Regiform', 
               'Mesuring', 'Wood Debris', 'Special Waste', 'Wood Trank']

# Aggregate by date, keeping waste_type
daily_data = df.groupby(['ticket_date', 'waste_type'])['net_weight_kg'].sum().unstack(fill_value=0).reset_index()
daily_data = daily_data.set_index('ticket_date')

# Add time-based features
daily_data['day_of_week'] = daily_data.index.dayofweek
daily_data['month'] = daily_data.index.month
daily_data['day_of_year'] = daily_data.index.dayofyear

# Total net_weight_kg
daily_data['total_weight'] = daily_data[waste_types].sum(axis=1)

# Outlier treatment using IQR method
def detect_outliers_iqr(data, column):
    Q1 = data[column].quantile(0.25)
    Q3 = data[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    return lower_bound, upper_bound

# Cap outliers
lower_iqr, upper_iqr = detect_outliers_iqr(daily_data, 'total_weight')
daily_data_cleaned = daily_data.copy()
daily_data_cleaned['total_weight'] = daily_data_cleaned['total_weight'].clip(lower=lower_iqr, upper=upper_iqr)

# Function to process data for a specific year
def process_year_data(year):
    # Filter data for the specified year
    daily_data_year = daily_data_cleaned[daily_data_cleaned.index.year == year].copy()
    
    # Enhance data with subtle variability (1% noise)
    np.random.seed(42)  # For reproducibility
    daily_data_year['total_weight_enhanced'] = daily_data_year['total_weight'] * (1 + np.random.normal(0, 0.01, daily_data_year.shape[0]))
    
    # Add lagged feature and moving average
    daily_data_year['lag_1'] = daily_data_year['total_weight_enhanced'].shift(1)
    daily_data_year['ma_7'] = daily_data_year['total_weight_enhanced'].rolling(window=7).mean()
    daily_data_year = daily_data_year.dropna()

    # Prepare features
    features = waste_types + ['day_of_week', 'month', 'day_of_year', 'total_weight_enhanced', 'lag_1', 'ma_7']
    data_year = daily_data_year[features].values
    scaler = StandardScaler()
    data_scaled_year = scaler.fit_transform(data_year)

    # Create sequences
    def create_sequences(data, seq_length):
        X, y = [], []
        for i in range(len(data) - seq_length):
            X.append(data[i:i + seq_length, :-1])  # All features except total_weight_enhanced
            y.append(data[i + seq_length, -1])  # Predict total_weight_enhanced
        return np.array(X), np.array(y)

    seq_length = 30
    X_year, y_year = create_sequences(data_scaled_year, seq_length)

    # Train-test split
    train_size = int(0.8 * len(X_year))
    X_train_year, X_test_year = X_year[:train_size], X_year[train_size:]
    y_train_year, y_test_year = y_year[:train_size], y_year[train_size:]
    test_dates_year = daily_data_year.index[seq_length + train_size:seq_length + len(X_year)]

    return X_train_year, X_test_year, y_train_year, y_test_year, test_dates_year, scaler, data_scaled_year.shape[1]

# Set the year to analyze
year_to_analyze = 2018

# Process data for the selected year
X_train_year, X_test_year, y_train_year, y_test_year, test_dates_year, scaler, n_features = process_year_data(year_to_analyze)

# Flatten sequence data for Linear Regression
X_train_flat = X_train_year.reshape(X_train_year.shape[0], -1)  # (samples, seq_length * (n_features-1))
X_test_flat = X_test_year.reshape(X_test_year.shape[0], -1)

# Custom Linear Regression with Gradient Descent and Gradient Clipping
class LinearRegressionGD:
    def __init__(self, learning_rate=0.01, n_iterations=1000, clip_value=1.0):
        self.learning_rate = learning_rate
        self.n_iterations = n_iterations
        self.clip_value = clip_value  # Gradient clipping threshold
        self.weights = None
        self.bias = None
        self.loss_history = []
        self.gradient_norms = []

    def fit(self, X, y):
        n_samples, n_features = X.shape
        self.weights = np.zeros(n_features)
        self.bias = 0

        for _ in range(self.n_iterations):
            # Forward pass
            y_pred = np.dot(X, self.weights) + self.bias

            # Compute gradients
            dw = (1 / n_samples) * np.dot(X.T, (y_pred - y))
            db = (1 / n_samples) * np.sum(y_pred - y)

            # Gradient clipping
            gradient_norm = np.sqrt(np.sum(dw ** 2) + db ** 2)
            if gradient_norm > self.clip_value:
                dw = dw * self.clip_value / gradient_norm
                db = db * self.clip_value / gradient_norm

            # Store clipped gradient norm
            self.gradient_norms.append(min(gradient_norm, self.clip_value))

            # Update parameters
            self.weights -= self.learning_rate * dw
            self.bias -= self.learning_rate * db

            # Compute MSE loss
            loss = np.mean((y_pred - y) ** 2)
            if np.isnan(loss):
                print(f"Warning: NaN loss detected at iteration {_}. Stopping training.")
                break
            self.loss_history.append(loss)

    def predict(self, X):
        return np.dot(X, self.weights) + self.bias

# Define learning rates to test (simulating optimizer variation)
learning_rates = [0.001, 0.01, 0.005]  # Replaced 0.1 with 0.005 for stability

# Train and evaluate Linear Regression with different learning rates
results = {}
for lr in learning_rates:
    print(f"\nTraining Linear Regression with learning rate {lr}...")
    model = LinearRegressionGD(learning_rate=lr, n_iterations=1000, clip_value=1.0)
    model.fit(X_train_flat, y_train_year)
    
    # Predict
    y_pred = model.predict(X_test_flat)
    
    # Inverse transform
    dummy_pred = np.zeros((len(y_pred), n_features))
    dummy_pred[:, -1] = y_pred
    y_pred_inv = scaler.inverse_transform(dummy_pred)[:, -1]
    
    dummy_test = np.zeros((len(y_test_year), n_features))
    dummy_test[:, -1] = y_test_year
    y_test_inv = scaler.inverse_transform(dummy_test)[:, -1]
    
    # Evaluate (skip if NaN detected)
    if np.any(np.isnan(y_pred_inv)):
        print(f"Warning: NaN predictions detected for lr={lr}. Skipping evaluation.")
        continue
    
    mae = mean_absolute_error(y_test_inv, y_pred_inv)
    rmse = np.sqrt(mean_squared_error(y_test_inv, y_pred_inv))
    print(f"Linear Regression (lr={lr}) - MAE: {mae:.2f} kg, RMSE: {rmse:.2f} kg")
    
    results[str(lr)] = {
        'model': model,
        'y_pred_inv': y_pred_inv,
        'mae': mae,
        'rmse': rmse
    }

# Visualizations
# 1. Actual vs Predicted for each learning rate
for lr in learning_rates:
    if str(lr) not in results:
        continue
    plt.figure(figsize=(12, 6))
    plt.plot(test_dates_year, y_test_inv, label='Actual Weight', color='blue')
    plt.plot(test_dates_year, results[str(lr)]['y_pred_inv'], 
             label=f'Predicted Weight (Linear Regression lr={lr})', color='green', linestyle='--')
    plt.xlabel('Date')
    plt.ylabel('Total Weight (kg)')
    plt.title(f'Actual vs Predicted Waste Weight (Linear Regression with lr={lr}) - {year_to_analyze}')
    plt.legend()
    plt.savefig(f'linear_regression_forecast_2018_lr_{str(lr).replace(".", "_")}.png')
    plt.show()

# 2. Training Loss Comparison
plt.figure(figsize=(12, 6))
for lr in learning_rates:
    if str(lr) not in results:
        continue
    plt.plot(results[str(lr)]['model'].loss_history, label=f'Training Loss (lr={lr})')
plt.xlabel('Iteration')
plt.ylabel('Mean Squared Error')
plt.title('Training Loss for Linear Regression with Different Learning Rates')
plt.legend()
plt.savefig('linear_regression_loss_comparison_2018.png')
plt.show()

# 3. Gradient Norm Comparison
plt.figure(figsize=(12, 6))
for lr in learning_rates:
    if str(lr) not in results:
        continue
    plt.plot(results[str(lr)]['model'].gradient_norms, label=f'Gradient Norm (lr={lr})')
plt.xlabel('Iteration')
plt.ylabel('Gradient Norm')
plt.title('Gradient Norm During Training for Linear Regression')
plt.legend()
plt.show()

<p style="font-family: Calibri, serif; text-align: left;
          font-size: 28px; letter-spacing: .85px; color: #ffffff;">LSTM Model with diffrent Optimizers</p>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dropout, Dense
from tensorflow.keras.optimizers import Adam, SGD, RMSprop, Adagrad

# Read data
df = pd.read_csv('dataset\moratuwa_2014-2018.csv')

# Preprocessing
df['ticket_date'] = pd.to_datetime(df['ticket_date'])
df = df.sort_values('ticket_date')

# Define waste types
waste_types = ['MSW', 'Sorted Organic Waste', 'Slaghter House Waste', 'Bulky Waste', 
               'Industrial Waste', 'Soil With Waste', 'Sanitary Waste', 'C&D Waste', 
               'Indutrial Sludge Waste', 'Soil', 'Saw Dust', 'Polythyne & Regiform', 
               'Mesuring', 'Wood Debris', 'Special Waste', 'Wood Trank']

# Aggregate by date, keeping waste_type
daily_data = df.groupby(['ticket_date', 'waste_type'])['net_weight_kg'].sum().unstack(fill_value=0).reset_index()
daily_data = daily_data.set_index('ticket_date')

# Add time-based features
daily_data['day_of_week'] = daily_data.index.dayofweek
daily_data['month'] = daily_data.index.month
daily_data['day_of_year'] = daily_data.index.dayofyear

# Total net_weight_kg
daily_data['total_weight'] = daily_data[waste_types].sum(axis=1)

# Outlier treatment using IQR method
def detect_outliers_iqr(data, column):
    Q1 = data[column].quantile(0.25)
    Q3 = data[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    return lower_bound, upper_bound

# Cap outliers
lower_iqr, upper_iqr = detect_outliers_iqr(daily_data, 'total_weight')
daily_data_cleaned = daily_data.copy()
daily_data_cleaned['total_weight'] = daily_data_cleaned['total_weight'].clip(lower=lower_iqr, upper=upper_iqr)

# Function to process data for a specific year
def process_year_data(year):
    # Filter data for the specified year
    daily_data_year = daily_data_cleaned[daily_data_cleaned.index.year == year].copy()
    
    # Enhance data with subtle variability (1% noise)
    np.random.seed(42)  # For reproducibility
    daily_data_year['total_weight_enhanced'] = daily_data_year['total_weight'] * (1 + np.random.normal(0, 0.01, daily_data_year.shape[0]))
    
    # Add lagged feature and moving average
    daily_data_year['lag_1'] = daily_data_year['total_weight_enhanced'].shift(1)
    daily_data_year['ma_7'] = daily_data_year['total_weight_enhanced'].rolling(window=7).mean()
    daily_data_year = daily_data_year.dropna()

    # Prepare features
    features = waste_types + ['day_of_week', 'month', 'day_of_year', 'total_weight_enhanced', 'lag_1', 'ma_7']
    data_year = daily_data_year[features].values
    scaler = StandardScaler()
    data_scaled_year = scaler.fit_transform(data_year)

    # Create sequences
    def create_sequences(data, seq_length):
        X, y = [], []
        for i in range(len(data) - seq_length):
            X.append(data[i:i + seq_length, :-1])  # All features except total_weight_enhanced
            y.append(data[i + seq_length, -1])  # Predict total_weight_enhanced
        return np.array(X), np.array(y)

    seq_length = 30
    X_year, y_year = create_sequences(data_scaled_year, seq_length)

    # Train-test split
    train_size = int(0.8 * len(X_year))
    X_train_year, X_test_year = X_year[:train_size], X_year[train_size:]
    y_train_year, y_test_year = y_year[:train_size], y_year[train_size:]
    test_dates_year = daily_data_year.index[seq_length + train_size:seq_length + len(X_year)]

    return X_train_year, X_test_year, y_train_year, y_test_year, test_dates_year, scaler, data_scaled_year.shape[1]

# Set the year to analyze
year_to_analyze = 2018

# Process data for the selected year
X_train_year, X_test_year, y_train_year, y_test_year, test_dates_year, scaler, n_features = process_year_data(year_to_analyze)

# Define optimizer configurations
optimizers = {
    'Adam': {'class': Adam, 'kwargs': {'learning_rate': 0.001}},
    'SGD': {'class': SGD, 'kwargs': {'learning_rate': 0.01, 'momentum': 0.9}},
    'RMSprop': {'class': RMSprop, 'kwargs': {'learning_rate': 0.001}},
    'Adagrad': {'class': Adagrad, 'kwargs': {'learning_rate': 0.01}}
}

# Function to create LSTM model
def create_lstm_model(optimizer):
    model = Sequential([
        LSTM(50, activation='tanh', input_shape=(30, X_train_year.shape[2]), return_sequences=False),
        Dropout(0.2),
        Dense(1)
    ])
    model.compile(optimizer=optimizer, loss='mse')
    return model

# Train and evaluate LSTM models with different optimizers
results = {}
for opt_name, opt_config in optimizers.items():
    print(f"\nTraining LSTM with {opt_name} optimizer...")
    # Create fresh optimizer instance
    optimizer = opt_config['class'](**opt_config['kwargs'])
    model = create_lstm_model(optimizer)
    history = model.fit(X_train_year, y_train_year, epochs=20, batch_size=32, validation_split=0.1, verbose=1)
    
    # Predict
    y_pred = model.predict(X_test_year)
    
    # Inverse transform
    dummy_pred = np.zeros((len(y_pred), n_features))
    dummy_pred[:, -1] = y_pred.flatten()
    y_pred_inv = scaler.inverse_transform(dummy_pred)[:, -1]
    
    dummy_test = np.zeros((len(y_test_year), n_features))
    dummy_test[:, -1] = y_test_year
    y_test_inv = scaler.inverse_transform(dummy_test)[:, -1]
    
    # Evaluate
    mae = mean_absolute_error(y_test_inv, y_pred_inv)
    rmse = np.sqrt(mean_squared_error(y_test_inv, y_pred_inv))
    print(f"LSTM ({opt_name}) - MAE: {mae:.2f} kg, RMSE: {rmse:.2f} kg")
    
    results[opt_name] = {
        'history': history.history,
        'y_pred_inv': y_pred_inv,
        'mae': mae,
        'rmse': rmse
    }

# Visualizations
# 1. Actual vs Predicted for each optimizer
for opt_name in optimizers.keys():
    plt.figure(figsize=(12, 6))
    plt.plot(test_dates_year, y_test_inv, label='Actual Weight', color='blue')
    plt.plot(test_dates_year, results[opt_name]['y_pred_inv'], 
             label=f'Predicted Weight (LSTM {opt_name})', color='orange', linestyle='--')
    plt.xlabel('Date')
    plt.ylabel('Total Weight (kg)')
    plt.title(f'Actual vs Predicted Waste Weight (LSTM with {opt_name}) - {year_to_analyze}')
    plt.legend()
    plt.savefig(f'lstm_forecast_2018_{opt_name.lower()}.png')
    plt.show()

# 2. Training and Validation Loss Comparison
plt.figure(figsize=(12, 6))
for opt_name in optimizers.keys():
    plt.plot(results[opt_name]['history']['loss'], 
             label=f'Training Loss ({opt_name})')
    plt.plot(results[opt_name]['history']['val_loss'], 
             label=f'Validation Loss ({opt_name})', linestyle='--')
plt.xlabel('Epoch')
plt.ylabel('Mean Squared Error')
plt.title('Training and Validation Loss for LSTM with Different Optimizers')
plt.legend()
plt.show()

<p style="font-family: Calibri, serif; text-align: left;
          font-size: 28px; letter-spacing: .85px; color: #ffffff;">Recurrent Neural Network (RNN) Model</p>

<p style="font-family: Calibri, serif; text-align: left;
          font-size: 24px; letter-spacing: .85px; color: #e74c3c;">The RNN model is a type of neural network designed for sequential data, making it suitable for time series forecasting. It processes the sequence of features to predict the next value, with dropout for regularization to prevent overfitting.</p>

<p style="font-family: Calibri, serif; text-align: left;
          font-size: 28px; letter-spacing: .85px; color: #ffffff;">Gated Recurrent Unit (GRU) Model</p>

<p style="font-family: Calibri, serif; text-align: left;
          font-size: 24px; letter-spacing: .85px; color: #e74c3c;">GRU is an advanced RNN variant that uses gating mechanisms to better handle long-term dependencies and is computationally efficient. It’s well-suited for time series with complex patterns like waste weight data.</p>

In [ ]:
%matplotlib inline
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import SimpleRNN, GRU, Dropout, Dense
from tensorflow.keras.optimizers import Adam, SGD, RMSprop, Adagrad

# Read data
df = pd.read_csv('dataset\moratuwa_2014-2018.csv')

# Preprocessing
df['ticket_date'] = pd.to_datetime(df['ticket_date'])
df = df.sort_values('ticket_date')

# Define waste types
waste_types = ['MSW', 'Sorted Organic Waste', 'Slaghter House Waste', 'Bulky Waste', 
               'Industrial Waste', 'Soil With Waste', 'Sanitary Waste', 'C&D Waste', 
               'Indutrial Sludge Waste', 'Soil', 'Saw Dust', 'Polythyne & Regiform', 
               'Mesuring', 'Wood Debris', 'Special Waste', 'Wood Trank']

# Aggregate by date, keeping waste_type
daily_data = df.groupby(['ticket_date', 'waste_type'])['net_weight_kg'].sum().unstack(fill_value=0).reset_index()
daily_data = daily_data.set_index('ticket_date')

# Add time-based features
daily_data['day_of_week'] = daily_data.index.dayofweek
daily_data['month'] = daily_data.index.month
daily_data['day_of_year'] = daily_data.index.dayofyear

# Total net_weight_kg
daily_data['total_weight'] = daily_data[waste_types].sum(axis=1)

# Outlier treatment using IQR method
def detect_outliers_iqr(data, column):
    Q1 = data[column].quantile(0.25)
    Q3 = data[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    return lower_bound, upper_bound

# Cap outliers
lower_iqr, upper_iqr = detect_outliers_iqr(daily_data, 'total_weight')
daily_data_cleaned = daily_data.copy()
daily_data_cleaned['total_weight'] = daily_data_cleaned['total_weight'].clip(lower=lower_iqr, upper=upper_iqr)

# Function to process data for a specific year
def process_year_data(daily_data_cleaned, years, seq_length=30):
    # Filter data for the specified years
    daily_data_year = daily_data_cleaned[daily_data_cleaned.index.year.isin(years)].copy()
    
    # Enhance data with subtle variability (1% noise)
    np.random.seed(42)  # For reproducibility
    daily_data_year['total_weight_enhanced'] = daily_data_year['total_weight'] * (1 + np.random.normal(0, 0.01, daily_data_year.shape[0]))
    
    # Add lagged feature and moving average
    daily_data_year['lag_1'] = daily_data_year['total_weight_enhanced'].shift(1)
    daily_data_year['ma_7'] = daily_data_year['total_weight_enhanced'].rolling(window=7).mean()
    daily_data_year = daily_data_year.dropna()

    # Prepare features
    features = waste_types + ['day_of_week', 'month', 'day_of_year', 'total_weight_enhanced', 'lag_1', 'ma_7']
    data_year = daily_data_year[features].values
    scaler = StandardScaler()
    data_scaled_year = scaler.fit_transform(data_year)

    # Create sequences
    def create_sequences(data, seq_length):
        X, y = [], []
        for i in range(len(data) - seq_length):
            X.append(data[i:i + seq_length, :-1])  # All features except total_weight_enhanced
            y.append(data[i + seq_length, -1])  # Predict total_weight_enhanced
        return np.array(X), np.array(y)

    X_year, y_year = create_sequences(data_scaled_year, seq_length)
    return X_year, y_year, scaler, data_scaled_year.shape[1]

# Process data for 2018
X_2018, y_2018, scaler_2018, n_features = process_year_data(daily_data_cleaned, years=[2018])
train_size = int(0.8 * len(X_2018))
X_train_2018, X_test_2018 = X_2018[:train_size], X_2018[train_size:]
y_train_2018, y_test_2018 = y_2018[:train_size], y_2018[train_size:]
test_dates_2018 = daily_data_cleaned[daily_data_cleaned.index.year == 2018].index[train_size + 30:len(X_2018) + 30]

# Define optimizer configurations
optimizers = {
    'Adam': {'class': Adam, 'kwargs': {'learning_rate': 0.001}},
    'SGD': {'class': SGD, 'kwargs': {'learning_rate': 0.01, 'momentum': 0.9}},
    'RMSprop': {'class': RMSprop, 'kwargs': {'learning_rate': 0.001}},
    'Adagrad': {'class': Adagrad, 'kwargs': {'learning_rate': 0.01}}
}

# Function to create GRU model
def create_gru_model():
    model = Sequential([
        GRU(50, activation='tanh', input_shape=(30, n_features-1), return_sequences=False),
        Dropout(0.2),
        Dense(1)
    ])
    return model

# Function to create RNN model
def create_rnn_model():
    model = Sequential([
        SimpleRNN(50, activation='tanh', input_shape=(30, n_features-1), return_sequences=False),
        Dropout(0.2),
        Dense(1)
    ])
    return model

# Models to process
models = {'GRU': create_gru_model, 'RNN': create_rnn_model}

# Train and evaluate each model
for model_name in models:
    print(f"\nTraining {model_name} on 2018 data...")
    results = {}
    for opt_name, opt_config in optimizers.items():
        print(f"\nTraining {model_name} with {opt_name} optimizer...")
        model = models[model_name]()
        optimizer = opt_config['class'](**opt_config['kwargs'])
        model.compile(optimizer=optimizer, loss='mse')
        
        # Train on 2018 data
        history = model.fit(X_train_2018, y_train_2018, epochs=10, batch_size=32, validation_split=0.1, verbose=1)
        
        # Predict
        y_pred = model.predict(X_test_2018)
        
        # Inverse transform
        dummy_pred = np.zeros((len(y_pred), n_features))
        dummy_pred[:, -1] = y_pred.flatten()
        y_pred_inv = scaler_2018.inverse_transform(dummy_pred)[:, -1]
        
        dummy_test = np.zeros((len(y_test_2018), n_features))
        dummy_test[:, -1] = y_test_2018
        y_test_inv = scaler_2018.inverse_transform(dummy_test)[:, -1]
        
        # Evaluate
        mae = mean_absolute_error(y_test_inv, y_pred_inv)
        rmse = np.sqrt(mean_squared_error(y_test_inv, y_pred_inv))
        print(f"{model_name} ({opt_name}) - MAE: {mae:.2f} kg, RMSE: {rmse:.2f} kg")
        
        results[opt_name] = {
            'history': history.history,
            'y_pred_inv': y_pred_inv,
            'mae': mae,
            'rmse': rmse
        }

    # Visualizations for the model
    # 1. Actual vs Predicted for each optimizer
    for opt_name in optimizers.keys():
        plt.figure(figsize=(12, 6))
        plt.plot(test_dates_2018, y_test_inv, label='Actual Weight', color='blue')
        plt.plot(test_dates_2018, results[opt_name]['y_pred_inv'], 
                 label=f'Predicted Weight ({model_name} {opt_name})', color='red', linestyle='--')
        plt.xlabel('Date')
        plt.ylabel('Total Weight (kg)')
        plt.title(f'Actual vs Predicted Waste Weight ({model_name} with {opt_name}) - 2018')
        plt.legend()
        plt.savefig(f'{model_name.lower()}_forecast_2018_{opt_name.lower()}.png')
        plt.show()

    # 2. Training and Validation Loss Comparison
    plt.figure(figsize=(12, 6))
    for opt_name in optimizers.keys():
        plt.plot(results[opt_name]['history']['loss'], 
                 label=f'Training Loss ({opt_name})')
        plt.plot(results[opt_name]['history']['val_loss'], 
                 label=f'Validation Loss ({opt_name})', linestyle='--')
    plt.xlabel('Epoch')
    plt.ylabel('Mean Squared Error')
    plt.title(f'Training Loss for {model_name} with Different Optimizers')
    plt.legend()
    plt.savefig(f'{model_name.lower()}_loss_comparison_2018.png')
    plt.show()

<p style="font-family: Calibri, serif; text-align: left;
          font-size: 28px; letter-spacing: .85px; color: #ffffff;">Hybrid LSTM-GRU Model</p>

<p style="font-family: Calibri, serif; text-align: left;
          font-size: 24px; letter-spacing: .85px; color: #e74c3c;">The Hybrid LSTM-GRU model combines the strengths of LSTM (long-term memory) and GRU (efficiency) to capture both long-term trends and short-term fluctuations. This makes it a powerful choice for complex time series like waste weight data with multiple features.</p>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, GRU, Dropout, Dense
from tensorflow.keras.optimizers import Adam, SGD, RMSprop, Adagrad

# Read data
df = pd.read_csv('dataset\moratuwa_2014-2018.csv')

# Preprocessing
df['ticket_date'] = pd.to_datetime(df['ticket_date'])
df = df.sort_values('ticket_date')

# Define waste types
waste_types = ['MSW', 'Sorted Organic Waste', 'Slaghter House Waste', 'Bulky Waste', 
               'Industrial Waste', 'Soil With Waste', 'Sanitary Waste', 'C&D Waste', 
               'Indutrial Sludge Waste', 'Soil', 'Saw Dust', 'Polythyne & Regiform', 
               'Mesuring', 'Wood Debris', 'Special Waste', 'Wood Trank']

# Aggregate by date, keeping waste_type
daily_data = df.groupby(['ticket_date', 'waste_type'])['net_weight_kg'].sum().unstack(fill_value=0).reset_index()
daily_data = daily_data.set_index('ticket_date')

# Add time-based features
daily_data['day_of_week'] = daily_data.index.dayofweek
daily_data['month'] = daily_data.index.month
daily_data['day_of_year'] = daily_data.index.dayofyear

# Total net_weight_kg
daily_data['total_weight'] = daily_data[waste_types].sum(axis=1)

# Outlier treatment using IQR method
def detect_outliers_iqr(data, column):
    Q1 = data[column].quantile(0.25)
    Q3 = data[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    return lower_bound, upper_bound

# Cap outliers
lower_iqr, upper_iqr = detect_outliers_iqr(daily_data, 'total_weight')
daily_data_cleaned = daily_data.copy()
daily_data_cleaned['total_weight'] = daily_data_cleaned['total_weight'].clip(lower=lower_iqr, upper=upper_iqr)

# Function to process data for a specific year
def process_year_data(year):
    # Filter data for the specified year
    daily_data_year = daily_data_cleaned[daily_data_cleaned.index.year == year].copy()
    
    # Enhance data with subtle variability (1% noise)
    np.random.seed(42)  # For reproducibility
    daily_data_year['total_weight_enhanced'] = daily_data_year['total_weight'] * (1 + np.random.normal(0, 0.01, daily_data_year.shape[0]))
    
    # Add lagged feature and moving average
    daily_data_year['lag_1'] = daily_data_year['total_weight_enhanced'].shift(1)
    daily_data_year['ma_7'] = daily_data_year['total_weight_enhanced'].rolling(window=7).mean()
    daily_data_year = daily_data_year.dropna()

    # Prepare features
    features = waste_types + ['day_of_week', 'month', 'day_of_year', 'total_weight_enhanced', 'lag_1', 'ma_7']
    data_year = daily_data_year[features].values
    scaler = StandardScaler()
    data_scaled_year = scaler.fit_transform(data_year)

    # Create sequences
    def create_sequences(data, seq_length):
        X, y = [], []
        for i in range(len(data) - seq_length):
            X.append(data[i:i + seq_length, :-1])  # All features except total_weight_enhanced
            y.append(data[i + seq_length, -1])     # Predict total_weight_enhanced
        return np.array(X), np.array(y)

    seq_length = 30
    X_year, y_year = create_sequences(data_scaled_year, seq_length)

    # Train-test split
    train_size = int(0.8 * len(X_year))
    X_train_year, X_test_year = X_year[:train_size], X_year[train_size:]
    y_train_year, y_test_year = y_year[:train_size], y_year[train_size:]
    test_dates_year = daily_data_year.index[seq_length + train_size:seq_length + len(X_year)]

    return X_train_year, X_test_year, y_train_year, y_test_year, test_dates_year, scaler, data_scaled_year.shape[1]

# Set the year to analyze
year_to_analyze = 2018

# Process data for the selected year
X_train_year, X_test_year, y_train_year, y_test_year, test_dates_year, scaler, n_features = process_year_data(year_to_analyze)

# Define optimizer configurations
optimizers = {
    'Adam': {'class': Adam, 'kwargs': {'learning_rate': 0.001}},
    'SGD': {'class': SGD, 'kwargs': {'learning_rate': 0.01, 'momentum': 0.9}},
    'RMSprop': {'class': RMSprop, 'kwargs': {'learning_rate': 0.001}},
    'Adagrad': {'class': Adagrad, 'kwargs': {'learning_rate': 0.01}}
}

# Function to create Hybrid LSTM-GRU model
def create_hybrid_model(optimizer):
    model = Sequential([
        LSTM(50, activation='tanh', input_shape=(seq_length, X_train_year.shape[2]), return_sequences=True),
        Dropout(0.2),
        GRU(50, activation='tanh', return_sequences=False),
        Dropout(0.2),
        Dense(1)
    ])
    model.compile(optimizer=optimizer, loss='mse')
    return model

# Train and evaluate Hybrid LSTM-GRU models with different optimizers
results = {}
for opt_name, opt_config in optimizers.items():
    print(f"\nTraining Hybrid LSTM-GRU with {opt_name} optimizer...")
    # Create fresh optimizer instance
    optimizer = opt_config['class'](**opt_config['kwargs'])
    model = create_hybrid_model(optimizer)
    history = model.fit(X_train_year, y_train_year, epochs=20, batch_size=32, validation_split=0.1, verbose=1)
    
    # Predict
    y_pred = model.predict(X_test_year)
    
    # Inverse transform
    dummy_pred = np.zeros((len(y_pred), n_features))
    dummy_pred[:, -1] = y_pred.flatten()
    y_pred_inv = scaler.inverse_transform(dummy_pred)[:, -1]
    
    dummy_test = np.zeros((len(y_test_year), n_features))
    dummy_test[:, -1] = y_test_year
    y_test_inv = scaler.inverse_transform(dummy_test)[:, -1]
    
    # Evaluate
    mae = mean_absolute_error(y_test_inv, y_pred_inv)
    rmse = np.sqrt(mean_squared_error(y_test_inv, y_pred_inv))
    print(f"Hybrid LSTM-GRU ({opt_name}) - MAE: {mae:.2f} kg, RMSE: {rmse:.2f} kg")
    
    results[opt_name] = {
        'history': history.history,
        'y_pred_inv': y_pred_inv,
        'mae': mae,
        'rmse': rmse
    }

# Visualizations
# 1. Actual vs Predicted for each optimizer
for opt_name in optimizers.keys():
    plt.figure(figsize=(12, 6))
    plt.plot(test_dates_year, y_test_inv, label='Actual Weight', color='blue')
    plt.plot(test_dates_year, results[opt_name]['y_pred_inv'], 
             label=f'Predicted Weight (Hybrid LSTM-GRU {opt_name})', color='red', linestyle='--')
    plt.xlabel('Date')
    plt.ylabel('Total Weight (kg)')
    plt.title(f'Actual vs Predicted Waste Weight (Hybrid LSTM-GRU with {opt_name}) - {year_to_analyze}')
    plt.legend()
    plt.savefig(f'hybrid_forecast_2018_{opt_name.lower()}.png')
    plt.show()

# 2. Training and Validation Loss Comparison
plt.figure(figsize=(12, 6))
for opt_name in optimizers.keys():
    plt.plot(results[opt_name]['history']['loss'], 
             label=f'Training Loss ({opt_name})')
    plt.plot(results[opt_name]['history']['val_loss'], 
             label=f'Validation Loss ({opt_name})', linestyle='--')
plt.xlabel('Epoch')
plt.ylabel('Mean Squared Error')
plt.title('Training and Validation Loss for Hybrid LSTM-GRU with Different Optimizers')
plt.legend()
plt.show()

<p style="font-family: Calibri, serif; text-align: left;
          font-size: 28px; letter-spacing: .85px; color: #ffffff;">Transfer learning for time series forecasting</p>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, GRU, Dropout, Dense
from tensorflow.keras.optimizers import Adam, SGD, RMSprop, Adagrad

# Read data
df = pd.read_csv('dataset\moratuwa_2014-2018.csv')

# Preprocessing
df['ticket_date'] = pd.to_datetime(df['ticket_date'])
df = df.sort_values('ticket_date')

# Define waste types
waste_types = ['MSW', 'Sorted Organic Waste', 'Slaghter House Waste', 'Bulky Waste', 
               'Industrial Waste', 'Soil With Waste', 'Sanitary Waste', 'C&D Waste', 
               'Indutrial Sludge Waste', 'Soil', 'Saw Dust', 'Polythyne & Regiform', 
               'Mesuring', 'Wood Debris', 'Special Waste', 'Wood Trank']

# Aggregate by date, keeping waste_type
daily_data = df.groupby(['ticket_date', 'waste_type'])['net_weight_kg'].sum().unstack(fill_value=0).reset_index()
daily_data = daily_data.set_index('ticket_date')

# Add time-based features
daily_data['day_of_week'] = daily_data.index.dayofweek
daily_data['month'] = daily_data.index.month
daily_data['day_of_year'] = daily_data.index.dayofyear

# Total net_weight_kg
daily_data['total_weight'] = daily_data[waste_types].sum(axis=1)

# Outlier treatment using IQR method
def detect_outliers_iqr(data, column):
    Q1 = data[column].quantile(0.25)
    Q3 = data[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    return lower_bound, upper_bound

# Cap outliers
lower_iqr, upper_iqr = detect_outliers_iqr(daily_data, 'total_weight')
daily_data_cleaned = daily_data.copy()
daily_data_cleaned['total_weight'] = daily_data_cleaned['total_weight'].clip(lower=lower_iqr, upper=upper_iqr)

# Function to process data for a specific year
def process_year_data(daily_data_cleaned, years, seq_length=30):
    # Filter data for the specified years
    daily_data_year = daily_data_cleaned[daily_data_cleaned.index.year.isin(years)].copy()
    
    # Enhance data with subtle variability (1% noise)
    np.random.seed(42)  # For reproducibility
    daily_data_year['total_weight_enhanced'] = daily_data_year['total_weight'] * (1 + np.random.normal(0, 0.01, daily_data_year.shape[0]))
    
    # Add lagged feature and moving average
    daily_data_year['lag_1'] = daily_data_year['total_weight_enhanced'].shift(1)
    daily_data_year['ma_7'] = daily_data_year['total_weight_enhanced'].rolling(window=7).mean()
    daily_data_year = daily_data_year.dropna()

    # Prepare features
    features = waste_types + ['day_of_week', 'month', 'day_of_year', 'total_weight_enhanced', 'lag_1', 'ma_7']
    data_year = daily_data_year[features].values
    scaler = StandardScaler()
    data_scaled_year = scaler.fit_transform(data_year)

    # Create sequences
    def create_sequences(data, seq_length):
        X, y = [], []
        for i in range(len(data) - seq_length):
            X.append(data[i:i + seq_length, :-1])  # All features except total_weight_enhanced
            y.append(data[i + seq_length, -1])  # Predict total_weight_enhanced
        return np.array(X), np.array(y)

    X_year, y_year = create_sequences(data_scaled_year, seq_length)
    return X_year, y_year, scaler, data_scaled_year.shape[1]

# Process data for pre-training (2014-2017)
X_pretrain, y_pretrain, _, _ = process_year_data(daily_data_cleaned, years=[2014, 2015, 2016, 2017])

# Process data for fine-tuning (2018)
X_2018, y_2018, scaler_2018, n_features = process_year_data(daily_data_cleaned, years=[2018])
train_size = int(0.8 * len(X_2018))
X_train_2018, X_test_2018 = X_2018[:train_size], X_2018[train_size:]
y_train_2018, y_test_2018 = y_2018[:train_size], y_2018[train_size:]
test_dates_2018 = daily_data_cleaned[daily_data_cleaned.index.year == 2018].index[train_size + 30:len(X_2018) + 30]

# Define optimizer configurations
optimizers = {
    'Adam': {'class': Adam, 'kwargs': {'learning_rate': 0.001}},
    'SGD': {'class': SGD, 'kwargs': {'learning_rate': 0.01, 'momentum': 0.9}},
    'RMSprop': {'class': RMSprop, 'kwargs': {'learning_rate': 0.001}},
    'Adagrad': {'class': Adagrad, 'kwargs': {'learning_rate': 0.01}}
}

# Function to create Hybrid LSTM-GRU model
def create_hybrid_model():
    model = Sequential([
        LSTM(50, activation='tanh', input_shape=(30, n_features-1), return_sequences=True),
        Dropout(0.2),
        GRU(50, activation='tanh', return_sequences=False),
        Dropout(0.2),
        Dense(1)
    ])
    return model

# Pre-train the model on 2014-2017 data
print("Pre-training Hybrid LSTM-GRU on 2014-2017 data...")
pretrain_model = create_hybrid_model()
pretrain_model.compile(optimizer=Adam(learning_rate=0.001), loss='mse')
pretrain_model.fit(X_pretrain, y_pretrain, epochs=20, batch_size=32, validation_split=0.1, verbose=1)

# Fine-tune and evaluate with different optimizers
results = {}
for opt_name, opt_config in optimizers.items():
    print(f"\nFine-tuning Hybrid LSTM-GRU with {opt_name} optimizer...")
    # Create a new model with pre-trained weights
    model = create_hybrid_model()
    model.set_weights(pretrain_model.get_weights())  # Transfer pre-trained weights
    optimizer = opt_config['class'](**opt_config['kwargs'])
    model.compile(optimizer=optimizer, loss='mse')
    
    # Fine-tune on 2018 data
    history = model.fit(X_train_2018, y_train_2018, epochs=10, batch_size=32, validation_split=0.1, verbose=1)
    
    # Predict
    y_pred = model.predict(X_test_2018)
    
    # Inverse transform
    dummy_pred = np.zeros((len(y_pred), n_features))
    dummy_pred[:, -1] = y_pred.flatten()
    y_pred_inv = scaler_2018.inverse_transform(dummy_pred)[:, -1]
    
    dummy_test = np.zeros((len(y_test_2018), n_features))
    dummy_test[:, -1] = y_test_2018
    y_test_inv = scaler_2018.inverse_transform(dummy_test)[:, -1]
    
    # Evaluate
    mae = mean_absolute_error(y_test_inv, y_pred_inv)
    rmse = np.sqrt(mean_squared_error(y_test_inv, y_pred_inv))
    print(f"Hybrid LSTM-GRU (Transfer Learning, {opt_name}) - MAE: {mae:.2f} kg, RMSE: {rmse:.2f} kg")
    
    results[opt_name] = {
        'history': history.history,
        'y_pred_inv': y_pred_inv,
        'mae': mae,
        'rmse': rmse
    }

# Visualizations
# 1. Actual vs Predicted for each optimizer
for opt_name in optimizers.keys():
    plt.figure(figsize=(12, 6))
    plt.plot(test_dates_2018, y_test_inv, label='Actual Weight', color='blue')
    plt.plot(test_dates_2018, results[opt_name]['y_pred_inv'], 
             label=f'Predicted Weight (Transfer Learning {opt_name})', color='red', linestyle='--')
    plt.xlabel('Date')
    plt.ylabel('Total Weight (kg)')
    plt.title(f'Actual vs Predicted Waste Weight (Transfer Learning with {opt_name}) - 2018')
    plt.legend()
    plt.savefig(f'transfer_learning_forecast_2018_{opt_name.lower()}.png')
    plt.show()

# 2. Training and Validation Loss Comparison
plt.figure(figsize=(12, 6))
for opt_name in optimizers.keys():
    plt.plot(results[opt_name]['history']['loss'], 
             label=f'Training Loss ({opt_name})')
    plt.plot(results[opt_name]['history']['val_loss'], 
             label=f'Validation Loss ({opt_name})', linestyle='--')
plt.xlabel('Epoch')
plt.ylabel('Mean Squared Error')
plt.title('Fine-Tuning Loss for Transfer Learning with Different Optimizers')
plt.legend()
plt.savefig('transfer_learning_loss_comparison_2018.png')
plt.show()

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np

# Data from the output
# Linear Regression metrics
lr_labels = ['LR (lr=0.001)', 'LR (lr=0.005)', 'LR (lr=0.01)']
lr_mae = [4969.42, 4474.47, 4313.96]
lr_rmse = [5752.06, 5385.89, 5453.44]

# LSTM metrics
lstm_labels = ['LSTM (Adam)', 'LSTM (SGD)', 'LSTM (RMSprop)', 'LSTM (Adagrad)']
lstm_mae = [5032.25, 4685.09, 4990.45, 4789.24]
lstm_rmse = [6356.78, 6099.91, 6428.05, 6195.54]

# LSTM training and validation loss (from output)
lstm_loss_data = {
    'Adam': {
        'train_loss': [1.2961, 0.8814, 0.6208, 0.4602, 0.3738, 0.2988, 0.2648, 0.2301, 0.1986, 0.1652, 
                       0.1761, 0.1525, 0.1497, 0.1366, 0.1276, 0.1286, 0.1212, 0.1263, 0.1214, 0.1248],
        'val_loss': [1.1505, 0.9870, 0.8120, 0.6359, 0.4931, 0.4234, 0.3747, 0.3315, 0.2992, 0.2674, 
                     0.2380, 0.2206, 0.2033, 0.1923, 0.1837, 0.1826, 0.1747, 0.1686, 0.1628, 0.1612]
    },
    'SGD': {
        'train_loss': [0.8625, 0.3426, 0.2942, 0.2087, 0.2026, 0.1882, 0.1829, 0.1846, 0.1701, 0.1637, 
                       0.1684, 0.1611, 0.1499, 0.1731, 0.1624, 0.1745, 0.1485, 0.1572, 0.1531, 0.1421],
        'val_loss': [0.7732, 0.3341, 0.2722, 0.2786, 0.2479, 0.2255, 0.2212, 0.2137, 0.2068, 0.2063, 
                     0.2067, 0.2059, 0.2006, 0.2030, 0.2005, 0.1906, 0.1832, 0.1889, 0.1948, 0.1773]
    },
    'RMSprop': {
        'train_loss': [0.8991, 0.4618, 0.3361, 0.2670, 0.2540, 0.1902, 0.1679, 0.1468, 0.1425, 0.1459, 
                       0.1339, 0.1320, 0.1265, 0.1218, 0.1118, 0.1131, 0.1142, 0.1165, 0.1110, 0.1176],
        'val_loss': [0.7565, 0.5195, 0.3925, 0.3281, 0.2820, 0.2375, 0.2025, 0.1746, 0.1628, 0.1488, 
                     0.1378, 0.1330, 0.1444, 0.1285, 0.1243, 0.1180, 0.1146, 0.1173, 0.1457, 0.1269]
    },
    'Adagrad': {
        'train_loss': [0.7234, 0.4148, 0.3123, 0.3044, 0.2632, 0.2346, 0.2205, 0.1971, 0.2103, 0.2087, 
                       0.1919, 0.1830, 0.1856, 0.1831, 0.1809, 0.1745, 0.1815, 0.1845, 0.1606, 0.1743],
        'val_loss': [0.5754, 0.4241, 0.3176, 0.2684, 0.2553, 0.2409, 0.2314, 0.2252, 0.2235, 0.2209, 
                     0.2166, 0.2185, 0.2129, 0.2139, 0.2186, 0.2122, 0.2087, 0.2110, 0.2052, 0.2066]
    }
}

# Combine labels and metrics for plotting
all_labels = lr_labels + lstm_labels
all_mae = lr_mae + lstm_mae
all_rmse = lr_rmse + lstm_rmse

# Plot 1: MAE Comparison
plt.figure(figsize=(12, 6))
bars = plt.bar(all_labels, all_mae, color=['#1f77b4']*3 + ['#ff7f0e']*4)
plt.xlabel('Model (Optimizer/Learning Rate)')
plt.ylabel('Mean Absolute Error (kg)')
plt.title('MAE Comparison: Linear Regression vs LSTM')
plt.xticks(rotation=45, ha='right')
for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, yval + 50, f'{yval:.2f}', ha='center', va='bottom')
plt.tight_layout()
plt.show()

# Plot 2: RMSE Comparison
plt.figure(figsize=(12, 6))
bars = plt.bar(all_labels, all_rmse, color=['#1f77b4']*3 + ['#ff7f0e']*4)
plt.xlabel('Model (Optimizer/Learning Rate)')
plt.ylabel('Root Mean Squared Error (kg)')
plt.title('RMSE Comparison: Linear Regression vs LSTM')
plt.xticks(rotation=45, ha='right')
for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, yval + 50, f'{yval:.2f}', ha='center', va='bottom')
plt.tight_layout()
plt.show()

# Plot 3: LSTM Training and Validation Loss
plt.figure(figsize=(12, 6))
epochs = range(1, 21)
for optimizer in lstm_loss_data:
    plt.plot(epochs, lstm_loss_data[optimizer]['train_loss'], label=f'{optimizer} Train Loss', linestyle='-')
    plt.plot(epochs, lstm_loss_data[optimizer]['val_loss'], label=f'{optimizer} Val Loss', linestyle='--')
plt.xlabel('Epoch')
plt.ylabel('Mean Squared Error')
plt.title('LSTM Training and Validation Loss by Optimizer')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np

# Data from the output
# RNN metrics
rnn_labels = ['RNN (Adam)', 'RNN (SGD)', 'RNN (RMSprop)', 'RNN (Adagrad)']
rnn_mae = [4743.74, 4473.05, 6163.79, 5482.97]
rnn_rmse = [6054.05, 6293.71, 7454.32, 8105.35]

# GRU metrics
gru_labels = ['GRU (Adam)', 'GRU (SGD)', 'GRU (RMSprop)', 'GRU (Adagrad)']
gru_mae = [5258.45, 4981.47, 4913.81, 5368.31]
gru_rmse = [6483.51, 6524.51, 6277.11, 6890.89]

# RNN training and validation loss
rnn_loss_data = {
    'Adam': {
        'train_loss': [2.2176, 1.3514, 0.8692, 0.7287, 0.5627, 0.4349, 0.4440, 0.3495, 0.3221, 0.2861],
        'val_loss': [1.3693, 1.0484, 0.8553, 0.7221, 0.6037, 0.5224, 0.4813, 0.4518, 0.4182, 0.3861]
    },
    'SGD': {
        'train_loss': [1.2899, 0.6566, 0.4630, 0.3025, 0.2233, 0.2104, 0.1547, 0.1290, 0.1186, 0.1058],
        'val_loss': [0.7238, 0.3395, 0.2294, 0.1349, 0.1232, 0.1065, 0.1402, 0.1478, 0.1345, 0.1180]
    },
    'RMSprop': {
        'train_loss': [1.5945, 1.0348, 0.7771, 0.5195, 0.3720, 0.3857, 0.3188, 0.2762, 0.2997, 0.2940],
        'val_loss': [1.2972, 1.0438, 0.8077, 0.5899, 0.4412, 0.3307, 0.2966, 0.2832, 0.2538, 0.2655]
    },
    'Adagrad': {
        'train_loss': [1.3483, 0.5326, 0.3996, 0.3857, 0.3335, 0.3262, 0.2890, 0.2786, 0.2368, 0.2564],
        'val_loss': [0.7627, 0.3901, 0.2707, 0.2457, 0.2585, 0.2375, 0.2190, 0.2096, 0.2319, 0.2047]
    }
}

# GRU training and validation loss
gru_loss_data = {
    'Adam': {
        'train_loss': [1.3173, 0.9241, 0.6492, 0.4629, 0.3362, 0.2578, 0.2266, 0.1951, 0.1886, 0.1701],
        'val_loss': [1.3078, 1.0360, 0.7572, 0.5535, 0.3788, 0.2603, 0.2099, 0.2195, 0.2167, 0.2013]
    },
    'SGD': {
        'train_loss': [0.8814, 0.4163, 0.4321, 0.3040, 0.2650, 0.2219, 0.1976, 0.1970, 0.1856, 0.1936],
        'val_loss': [0.4444, 0.7054, 0.3957, 0.3381, 0.3299, 0.3237, 0.2718, 0.2617, 0.2500, 0.2271]
    },
    'RMSprop': {
        'train_loss': [0.6350, 0.3927, 0.2903, 0.2409, 0.2361, 0.1924, 0.1926, 0.1936, 0.1655, 0.1755],
        'val_loss': [0.5066, 0.4089, 0.2955, 0.2569, 0.2425, 0.2280, 0.2164, 0.1967, 0.2097, 0.1789]
    },
    'Adagrad': {
        'train_loss': [0.9475, 0.5089, 0.4124, 0.3639, 0.3331, 0.3427, 0.3020, 0.2598, 0.2472, 0.2468],
        'val_loss': [0.4311, 0.3871, 0.3840, 0.3770, 0.3341, 0.3421, 0.3344, 0.3152, 0.2923, 0.2793]
    }
}

# Combine labels and metrics for plotting
all_labels = rnn_labels + gru_labels
all_mae = rnn_mae + gru_mae
all_rmse = rnn_rmse + gru_rmse

# Plot 1: MAE Comparison
plt.figure(figsize=(12, 6))
bars = plt.bar(all_labels, all_mae, color=['#1f77b4']*4 + ['#ff7f0e']*4)
plt.xlabel('Model (Optimizer)')
plt.ylabel('Mean Absolute Error (kg)')
plt.title('MAE Comparison: RNN vs GRU')
plt.xticks(rotation=45, ha='right')
for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, yval + 50, f'{yval:.2f}', ha='center', va='bottom')
plt.tight_layout()
plt.savefig('rnn_gru_mae_comparison.png')
plt.show()

# Plot 2: RMSE Comparison
plt.figure(figsize=(12, 6))
bars = plt.bar(all_labels, all_rmse, color=['#1f77b4']*4 + ['#ff7f0e']*4)
plt.xlabel('Model (Optimizer)')
plt.ylabel('Root Mean Squared Error (kg)')
plt.title('RMSE Comparison: RNN vs GRU')
plt.xticks(rotation=45, ha='right')
for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, yval + 50, f'{yval:.2f}', ha='center', va='bottom')
plt.tight_layout()
plt.savefig('rnn_gru_rmse_comparison.png')
plt.show()

# Plot 3: RNN Training and Validation Loss
plt.figure(figsize=(12, 6))
epochs = range(1, 11)
for optimizer in rnn_loss_data:
    plt.plot(epochs, rnn_loss_data[optimizer]['train_loss'], label=f'RNN {optimizer} Train Loss', linestyle='-')
    plt.plot(epochs, rnn_loss_data[optimizer]['val_loss'], label=f'RNN {optimizer} Val Loss', linestyle='--')
plt.xlabel('Epoch')
plt.ylabel('Mean Squared Error')
plt.title('RNN Training and Validation Loss by Optimizer')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig('rnn_loss_comparison.png')
plt.show()

# Plot 4: GRU Training and Validation Loss
plt.figure(figsize=(12, 6))
for optimizer in gru_loss_data:
    plt.plot(epochs, gru_loss_data[optimizer]['train_loss'], label=f'GRU {optimizer} Train Loss', linestyle='-')
    plt.plot(epochs, gru_loss_data[optimizer]['val_loss'], label=f'GRU {optimizer} Val Loss', linestyle='--')
plt.xlabel('Epoch')
plt.ylabel('Mean Squared Error')
plt.title('GRU Training and Validation Loss by Optimizer')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig('gru_loss_comparison.png')
plt.show()

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np

# Data from the output
# Hybrid LSTM-GRU metrics (without transfer learning)
direct_labels = ['Hybrid LSTM-GRU (Adam)', 'Hybrid LSTM-GRU (SGD)', 'Hybrid LSTM-GRU (RMSprop)', 'Hybrid LSTM-GRU (Adagrad)']
direct_mae = [4437.08, 4551.78, 4565.17, 5023.78]
direct_rmse = [5464.32, 5675.24, 5959.29, 6113.30]

# Hybrid LSTM-GRU metrics (with transfer learning)
tl_labels = ['Hybrid LSTM-GRU (TL, Adam)', 'Hybrid LSTM-GRU (TL, SGD)', 'Hybrid LSTM-GRU (TL, RMSprop)', 'Hybrid LSTM-GRU (TL, Adagrad)']
tl_mae = [4015.80, 4174.78, 4129.25, 4172.45]
tl_rmse = [5337.70, 5704.62, 5564.00, 5646.03]

# Hybrid LSTM-GRU training and validation loss (without transfer learning)
direct_loss_data = {
    'Adam': {
        'train_loss': [0.8893, 0.4549, 0.3220, 0.2358, 0.1919, 0.1729, 0.1640, 0.1512, 0.1507, 0.1459, 
                       0.1467, 0.1356, 0.1473, 0.1364, 0.1404, 0.1184, 0.1347, 0.1278, 0.1130, 0.1175],
        'val_loss': [0.6605, 0.4743, 0.3635, 0.2849, 0.2154, 0.1912, 0.1827, 0.1821, 0.1758, 0.1927, 
                     0.1836, 0.1656, 0.1472, 0.1427, 0.1445, 0.1383, 0.1327, 0.1320, 0.1171, 0.1239]
    },
    'SGD': {
        'train_loss': [1.0209, 0.5011, 0.3341, 0.2576, 0.2188, 0.2032, 0.2004, 0.1769, 0.1723, 0.1560, 
                       0.1535, 0.1563, 0.1699, 0.1507, 0.1480, 0.1568, 0.1508, 0.1349, 0.1483, 0.1557],
        'val_loss': [0.8356, 0.2594, 0.2665, 0.2165, 0.2188, 0.1760, 0.1278, 0.1165, 0.1083, 0.1048, 
                     0.1132, 0.0998, 0.0981, 0.1049, 0.1032, 0.0963, 0.0969, 0.0930, 0.0917, 0.1043]
    },
    'RMSprop': {
        'train_loss': [0.8127, 0.4758, 0.3286, 0.3012, 0.2071, 0.1847, 0.1919, 0.1670, 0.1569, 0.1449, 
                       0.1447, 0.1613, 0.1486, 0.1415, 0.1402, 0.1231, 0.1296, 0.1258, 0.1275, 0.1366],
        'val_loss': [0.6976, 0.5120, 0.3717, 0.2592, 0.2072, 0.1685, 0.1382, 0.1400, 0.1337, 0.1253, 
                     0.1101, 0.1497, 0.1928, 0.1703, 0.1370, 0.1823, 0.1287, 0.1180, 0.1420, 0.1440]
    },
    'Adagrad': {
        'train_loss': [0.9442, 0.6079, 0.4356, 0.3343, 0.2824, 0.2421, 0.2517, 0.2239, 0.2009, 0.2156, 
                       0.2257, 0.1983, 0.1711, 0.1895, 0.1942, 0.1742, 0.1857, 0.1830, 0.1911, 0.1691],
        'val_loss': [0.9087, 0.7553, 0.4465, 0.3257, 0.2803, 0.2539, 0.2424, 0.2277, 0.2290, 0.2166, 
                     0.2225, 0.2227, 0.2381, 0.2162, 0.2073, 0.2076, 0.2151, 0.1750, 0.1869, 0.1837]
    }
}

# Hybrid LSTM-GRU training and validation loss (with transfer learning, fine-tuning)
tl_loss_data = {
    'Adam': {
        'train_loss': [0.2517, 0.1849, 0.1691, 0.1365, 0.1483, 0.1237, 0.1249, 0.1137, 0.1152, 0.1207],
        'val_loss': [0.1398, 0.1108, 0.1095, 0.1110, 0.1122, 0.1201, 0.1099, 0.1026, 0.0983, 0.0980]
    },
    'SGD': {
        'train_loss': [0.2854, 0.1895, 0.1652, 0.1549, 0.1514, 0.1404, 0.1426, 0.1435, 0.1386, 0.1357],
        'val_loss': [0.1682, 0.1352, 0.1149, 0.1110, 0.1129, 0.1084, 0.1093, 0.1042, 0.1072, 0.1042]
    },
    'RMSprop': {
        'train_loss': [0.2212, 0.1546, 0.1413, 0.1407, 0.1367, 0.1336, 0.1210, 0.1237, 0.1266, 0.1149],
        'val_loss': [0.1164, 0.1100, 0.1261, 0.1187, 0.1085, 0.1330, 0.0983, 0.1075, 0.1129, 0.1075]
    },
    'Adagrad': {
        'train_loss': [0.2736, 0.1937, 0.1745, 0.1617, 0.1592, 0.1709, 0.1502, 0.1376, 0.1629, 0.1440],
        'val_loss': [0.1844, 0.1448, 0.1212, 0.1335, 0.1180, 0.1128, 0.1076, 0.1102, 0.1129, 0.1130]
    }
}

# Pre-training loss data (2014-2017)
pretrain_loss_data = {
    'train_loss': [0.3246, 0.1126, 0.0855, 0.0831, 0.0729, 0.0728, 0.0674, 0.0656, 0.0645, 0.0661, 
                   0.0601, 0.0607, 0.0608, 0.0564, 0.0551, 0.0560, 0.0592, 0.0560, 0.0565, 0.0523],
    'val_loss': [0.1266, 0.0883, 0.0761, 0.0801, 0.0767, 0.0851, 0.0804, 0.0885, 0.0808, 0.0948, 
                 0.0802, 0.0902, 0.0788, 0.0959, 0.0830, 0.0765, 0.0804, 0.0761, 0.0795, 0.0820]
}

# Combine labels and metrics for plotting
all_labels = direct_labels + tl_labels
all_mae = direct_mae + tl_mae
all_rmse = direct_rmse + tl_rmse

# Plot 1: MAE Comparison
plt.figure(figsize=(12, 6))
bars = plt.bar(all_labels, all_mae, color=['#1f77b4']*4 + ['#ff7f0e']*4)
plt.xlabel('Model (Optimizer)')
plt.ylabel('Mean Absolute Error (kg)')
plt.title('MAE Comparison: Hybrid LSTM-GRU (Direct vs Transfer Learning)')
plt.xticks(rotation=45, ha='right')
for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, yval + 50, f'{yval:.2f}', ha='center', va='bottom')
plt.tight_layout()
plt.savefig('hybrid_lstm_gru_mae_comparison.png')
plt.show()

# Plot 2: RMSE Comparison
plt.figure(figsize=(12, 6))
bars = plt.bar(all_labels, all_rmse, color=['#1f77b4']*4 + ['#ff7f0e']*4)
plt.xlabel('Model (Optimizer)')
plt.ylabel('Root Mean Squared Error (kg)')
plt.title('RMSE Comparison: Hybrid LSTM-GRU (Direct vs Transfer Learning)')
plt.xticks(rotation=45, ha='right')
for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, yval + 50, f'{yval:.2f}', ha='center', va='bottom')
plt.tight_layout()
plt.savefig('hybrid_lstm_gru_rmse_comparison.png')
plt.show()


# Plot 3: Hybrid LSTM-GRU Training and Validation Loss (Direct Training)
plt.figure(figsize=(12, 6))
epochs_direct = range(1, 21)
for optimizer in direct_loss_data:
    plt.plot(epochs_direct, direct_loss_data[optimizer]['train_loss'], label=f'Direct {optimizer} Train Loss', linestyle='-')
    plt.plot(epochs_direct, direct_loss_data[optimizer]['val_loss'], label=f'Direct {optimizer} Val Loss', linestyle='--')
plt.xlabel('Epoch')
plt.ylabel('Mean Squared Error')
plt.title('Hybrid LSTM-GRU Training and Validation Loss (Direct Training) by Optimizer')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig('hybrid_lstm_gru_direct_loss_comparison.png')
plt.show()

# Plot 4: Hybrid LSTM-GRU Training and Validation Loss (Transfer Learning Fine-Tuning)
plt.figure(figsize=(12, 6))
epochs_tl = range(1, 11)
for optimizer in tl_loss_data:
    plt.plot(epochs_tl, tl_loss_data[optimizer]['train_loss'], label=f'TL {optimizer} Train Loss', linestyle='-')
    plt.plot(epochs_tl, tl_loss_data[optimizer]['val_loss'], label=f'TL {optimizer} Val Loss', linestyle='--')
plt.xlabel('Epoch')
plt.ylabel('Mean Squared Error')
plt.title('Hybrid LSTM-GRU Training and Validation Loss (Transfer Learning Fine-Tuning) by Optimizer')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig('hybrid_lstm_gru_tl_loss_comparison.png')
plt.show()

# Plot 5: Pre-training Loss (2014-2017)
plt.figure(figsize=(12, 6))
epochs_pretrain = range(1, 21)
plt.plot(epochs_pretrain, pretrain_loss_data['train_loss'], label='Pre-training Train Loss', linestyle='-')
plt.plot(epochs_pretrain, pretrain_loss_data['val_loss'], label='Pre-training Val Loss', linestyle='--')
plt.xlabel('Epoch')
plt.ylabel('Mean Squared Error')
plt.title('Hybrid LSTM-GRU Pre-training Loss (2014-2017)')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig('hybrid_lstm_gru_pretrain_loss.png')
plt.show()